In [1]:
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, RobustScaler

sys.path.append(str(Path.cwd().parent))

from src.preprocessing import create_new_features

In [2]:
train = pd.read_csv("../data/processed/train.csv")
val = pd.read_csv("../data/processed/val.csv")
test = pd.read_csv("../data/processed/test.csv")

In [3]:
sample_features = create_new_features(train.iloc[:5])
cols_to_scale = [col for col in sample_features.columns.to_list() if col not in
                 ["Hour_sin","Hour_cos","Class"]]

X_train = train.drop(columns=["Class"])
y_train = train["Class"]

X_val = val.drop(columns=["Class"])
y_val = val["Class"]

X_test = test.drop(columns=["Class"])
y_test = test["Class"]

transformer = ColumnTransformer(
    transformers=[
        ("robust_scaling", RobustScaler(), cols_to_scale)
    ],
    remainder="passthrough"
)

transformer.set_output(transform="pandas")

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('robust_scaling', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_na

In [4]:
preprocessing_pipeline = Pipeline(
    [
    ("feature_engineering", FunctionTransformer(create_new_features)),
    ("scaling", transformer)
    ]
)



X_train = preprocessing_pipeline.fit_transform(X_train)
X_val = preprocessing_pipeline.transform(X_val)
X_test = preprocessing_pipeline.transform(X_test)

X_train.columns = [col.split("__")[-1] for col in X_train.columns]
X_val.columns = [col.split("__")[-1] for col in X_val.columns]
X_test.columns = [col.split("__")[-1] for col in X_test.columns]

In [5]:
print(X_train.describe())

                  V1             V2             V3             V4  \
count  198277.000000  198277.000000  198277.000000  198277.000000   
mean       -0.004001      -0.050485      -0.090215       0.009959   
std         0.860440       1.162391       0.762747       0.880866   
min       -18.157575     -45.270718     -25.342989      -3.518568   
25%        -0.419444      -0.475121      -0.558843      -0.519533   
50%         0.000000       0.000000       0.000000       0.000000   
75%         0.580556       0.524879       0.441157       0.480467   
max         1.091211      13.451819       4.805067      10.573458   

                  V5             V6             V7             V8  \
count  198277.000000  198277.000000  198277.000000  198277.000000   
mean        0.044797       0.235298      -0.027070      -0.044514   
std         1.042942       1.143267       1.054903       2.170001   
min       -87.131864     -19.939280     -23.643933     -95.213097   
25%        -0.487116      -0.4251

In [6]:
datasets = {
    "X_train": X_train,
    "X_val": X_val,
    "X_test": X_test,
    "y_train": y_train,
    "y_val": y_val,
    "y_test": y_test,
}

output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)
for name, df in datasets.items():
    df.to_csv(output_dir/f"{name}.csv", index=False)

In [7]:
output_dir = Path("../models")
output_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(preprocessing_pipeline, "../models/preprocessing_pipeline.joblib")

['../models/preprocessing_pipeline.joblib']